### Online Retail II — Customer-Month Feature Table & Churn Labeling

Takes the v3 cleaned dataset (`data/online_retail_ii_cleaned_v3.parquet`) and prepares it for Exploratory Data Analysis (EDA) (Task #5):

1. Load + inspect standardized data
2. Aggregate customer-month features: purchase frequency, total spend, average order value, unique items purchases
3. Compute days-since-last-purchase using only prior data
4. Apply the forward-looking 90-day churn label
5. Exclude customer-months without a full, observable 90-day window
6. Save the customer-month feature table

## 1. Load cleaned data

In [6]:
# imports
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_parquet('../data/online_retail_ii_cleaned_v3.parquet')
print(df.shape)

# cancellations/returns are not purchases: drop them so they can't inflate order counts,
# unique items, or be mistaken for a "return purchase" in the churn label
print(f"Cancellation rows removed: {df['Is_Cancellation'].sum():,}")
df = df[~df['Is_Cancellation']].copy()
print("Purchases-only shape:", df.shape)

# columns to use
df[['Invoice', 'Customer ID', 'InvoiceDate', 'Quantity', 'Price', 'Total_Price', 'StockCode', 'CustomerMonth']].head()


(794183, 12)
Cancellation rows removed: 17,587
Purchases-only shape: (776596, 12)


,Invoice,Customer ID,InvoiceDate,Quantity,Price,Total_Price,StockCode,CustomerMonth
0,499763,12346,2010-03-02 13:08:00,1,3.25,3.25,20682,2010-03
1,499763,12346,2010-03-02 13:08:00,1,5.95,5.95,20679,2010-03
2,499763,12346,2010-03-02 13:08:00,1,5.95,5.95,15056N,2010-03
3,499763,12346,2010-03-02 13:08:00,1,5.95,5.95,15056BL,2010-03
4,499763,12346,2010-03-02 13:08:00,1,5.95,5.95,15056P,2010-03


## 2. Aggregate customer-month features

For each customer-month, only transactions within that month are used to compute purchase frequency, total spend, average order value, and unique items

In [7]:
monthly = (
    df.groupby(['Customer ID', 'CustomerMonth'])
    .agg(
        purchase_frequency = ('Invoice', 'nunique'),
        total_spend = ('Total_Price', 'sum'),
        unique_items = ('StockCode', 'nunique'),
        first_purchase_date = ('InvoiceDate', 'min'),
        last_purchase_date = ('InvoiceDate', 'max')
    )
    .reset_index()
    .sort_values(['Customer ID', 'CustomerMonth'])
    .reset_index(drop=True)
)

monthly['avg_order_value'] = monthly['total_spend'] / monthly['purchase_frequency']

print(f"Customer-month rows: {len(monthly):,}")
print(f"Unique customers: {monthly['Customer ID'].nunique():,}")
monthly.head()

Customer-month rows: 25,503
Unique customers: 5,852


,Customer ID,CustomerMonth,purchase_frequency,total_spend,unique_items,first_purchase_date,last_purchase_date,avg_order_value
0,12346,2010-03,1,27.05,5,2010-03-02 13:08:00,2010-03-02 13:08:00,27.05
1,12346,2010-06,1,142.31,19,2010-06-28 13:53:00,2010-06-28 13:53:00,142.31
2,12346,2011-01,1,149.76,1,2011-01-18 10:01:00,2011-01-18 10:01:00,149.76
3,12347,2010-10,1,611.53,40,2010-10-31 14:20:00,2010-10-31 14:20:00,611.53
4,12347,2010-12,1,711.79,31,2010-12-07 14:57:00,2010-12-07 14:57:00,711.79


## 3. Days since last purchase 
Recency is measured as of the start of each customer-month, using only the customer's purchase history strictly before this month. A customer's very first observed month has no prior purchase, so recency is left as `NaN` and flagged with `is_first_month` instead of being guessed at.

In [8]:
# most recent purchase date from the customer's previous active month
monthly['prev_month_last_purchase'] = (
    monthly.groupby('Customer ID')['last_purchase_date'].shift(1)
)

monthly['days_since_last_purchase'] = (
    monthly['first_purchase_date'] - monthly['prev_month_last_purchase']
).dt.days

monthly['is_first_month'] = monthly['prev_month_last_purchase'].isna()

print(f"First-month rows (no prior purchase, recency = NaN): {monthly['is_first_month'].sum():,}")
monthly[['Customer ID', 'CustomerMonth', 'first_purchase_date', 'prev_month_last_purchase',
         'days_since_last_purchase', 'is_first_month']].head()


First-month rows (no prior purchase, recency = NaN): 5,852


,Customer ID,CustomerMonth,first_purchase_date,prev_month_last_purchase,days_since_last_purchase,is_first_month
0,12346,2010-03,2010-03-02 13:08:00,NaT,NaN,True
1,12346,2010-06,2010-06-28 13:53:00,2010-03-02 13:08:00,118.0,False
2,12346,2011-01,2011-01-18 10:01:00,2010-06-28 13:53:00,203.0,False
3,12347,2010-10,2010-10-31 14:20:00,NaT,NaN,True
4,12347,2010-12,2010-12-07 14:57:00,2010-10-31 14:20:00,37.0,False


## 4. Forward-looking 90-day churn label

A customer-month is labeled churnned (`is_churn=1`) if there's no purchase in the 90 days after the customer's last purchase in that month.
We have to be careful: if we don't see a next purchase in 90 days, that could mean the customer truly went quiet OR it could mean the dataset ends before we had the chance to see them come back.
We only assign `1` label when the dataset actually extends far enought past `last_purchase_date` to have observed a would-be return purchase. 
A confirmed return purchase within 90 days results in `is_churn=0`

In [9]:
# every unique (Customer ID, InvoiceDate) purchase event, used to look up each customer's *next* purchase
purchase_events = (
    df[['Customer ID', 'InvoiceDate']]
    .drop_duplicates() 
    .sort_values('InvoiceDate')
    .reset_index(drop=True)
)

# for each customer-month's last purchase, find the very next purchase date after it (allow_exact_matches=False
# means same-timestamp invoices don't count, so this only looks strictly forward in time)
lookup = monthly[['Customer ID', 'last_purchase_date']].sort_values('last_purchase_date').reset_index(drop=True)

# merge_asof with direction='forward' finds, for each row, the nearest purchase date AFTER last_purchase_date
# for the SAME customer ('by'). allow_exact_matches=False means same-timestamp invoices don't count,
# so it only looks strictly forward in time
next_purchase = pd.merge_asof(
    lookup,
    purchase_events.rename(columns={'InvoiceDate': 'next_purchase_date'}),
    left_on='last_purchase_date',
    right_on='next_purchase_date',
    by='Customer ID',
    direction='forward',
    allow_exact_matches=False,
)

# attach next_purchase_date back onto the customer-month table
monthly = monthly.merge(
    next_purchase[['Customer ID', 'last_purchase_date', 'next_purchase_date']],
    on=['Customer ID', 'last_purchase_date'],
    how='left',
)


# days until the customer's next purchase (NaN if they never purchase again in the data)
monthly['days_to_next_purchase'] = (
    monthly['next_purchase_date'] - monthly['last_purchase_date']
).dt.days

# the last date we actually have data for
dataset_max_date = df['InvoiceDate'].max()
print(f"Dataset Max InvoiceDate: {dataset_max_date}")


# the 90-day window runs from last_purchase_date to window_end
# It is "observable" only if the data reaches at least that far
monthly['window_end'] = monthly['last_purchase_date'] + pd.Timedelta(days=90)
monthly['window_observable'] = monthly['window_end'] <= dataset_max_date


# True where we saw the customer come back within 90 days (NaN comparisons evaluate to False)
returned_within_90d = monthly['days_to_next_purchase'].le(90)

# if the full 90-day window isn't observable, the row is unknown (NaN)
# even if we happened to see a return purchase
# Keeping only "returned" customers near the end would leave those months with almost no label-1 rows 

# window fully visible + returned within 90d -> 0
# window fully visible + no return           -> 1
# window cut off by end of data              -> NaN (dropped in step 5)
monthly['is_churn'] = np.where(
    monthly['window_observable'],
    np.where(returned_within_90d, 0, 1),
    np.nan
)

print(monthly['is_churn'].value_counts(dropna=False))

Dataset Max InvoiceDate: 2011-12-09 12:50:00
is_churn
0.0    12738
1.0     8142
NaN     4623
Name: count, dtype: int64


## 5. Exclude customer-months without a full 90-day forward window

Rows where `is_churn` is `NaN` sit near the end of the dataset: there's no confirmed return purchase, but the data also doesn't run far enough past `last_purchase_date` to be sure the customer wouldn't have come back.
Keeping them would force a guess. Labeling them `1` would inflate the churn rate with customers who may simply not have had time to return, and labeling them `0` would understate it. Dropping them keeps the labels honest, at the cost of losing the most recent months.

In [10]:
before = len(monthly) # track amount of customers before

# keep only rows with known label, drop the NaN
monthly = monthly[monthly['is_churn'].notna()].copy()

# no more NaN, is_churn can be a clean integer column
monthly['is_churn'] = monthly['is_churn'].astype(int)

after = len(monthly) # track amount of customers after

## info
print(f"Dropped {before - after:,} customer-months with an unobservable 90-day window")
print(f"Remaining customer-months: {after:,}")
print(f"Latest remaining CustomerMonth: {monthly['CustomerMonth'].max()}")

# share of customer-months labeled at-risk; useful context for the Layer 2 models (class balance)
print(f"Churn rate: {monthly['is_churn'].mean():.2%}")

Dropped 4,623 customer-months with an unobservable 90-day window
Remaining customer-months: 20,880
Latest remaining CustomerMonth: 2011-09
Churn rate: 38.99%


## 6 Checking Before Saving

- No unexpected nulls (the only allowed nulls are the intentional first-month gaps in recency)
- `avg_order_value` reconstructs `total_spend` when multiplied back by `purchase_frequency`
- No customer-month contains a purchase date outside its own calendary month

In [11]:
columns_to_check = ['purchase_frequency', 'total_spend', 'avg_order_value', 'unique_items', 'is_churn']

print("Null in core feature columns:")
print(monthly[columns_to_check].isna().sum())

# recency should be missing exactly the first-month rows and nowhere else
print("\nNulls in days_since_last_purchase:")
print(
    monthly['days_since_last_purchase'].isna().sum(), "vs is_first_month sum:", monthly['is_first_month'].sum())

# total_spend should =  avg_order_value * purchase_frequency
reconstruct = np.allclose(
    monthly['avg_order_value'] * monthly['purchase_frequency'], monthly['total_spend']

)
print("\navg_order_value reconstructs total_spend:", reconstruct)

# every purchase date used for a customer-month should fall in that month
month_mismatch = (
    monthly['first_purchase_date'].dt.to_period('M') != monthly['CustomerMonth']
).sum()
print("\nCustomer-months with a purchase date outside their month:", month_mismatch)


Null in core feature columns:
purchase_frequency    0
total_spend           0
avg_order_value       0
unique_items          0
is_churn              0
dtype: int64

Nulls in days_since_last_purchase:
5248 vs is_first_month sum: 5248

avg_order_value reconstructs total_spend: True

Customer-months with a purchase date outside their month: 0


## 7. Assemble & Save final feature table

Keep only the columns needed so helper columns (purchase dates, window_end, etc.) are dropped so the table contains just identifiers, features, and the churn label

In [12]:
feature_cols = [
    'Customer ID', 'CustomerMonth', 
    'purchase_frequency', 'total_spend', 'avg_order_value', 'unique_items','days_since_last_purchase', 'is_first_month',
    'is_churn'

]

customer_month_features = monthly[feature_cols].reset_index(drop=True)

# final output yay!
print("Final Shape:", customer_month_features.shape)
print(customer_month_features.dtypes)
customer_month_features.head(10)

Final Shape: (20880, 9)
Customer ID                     int64
CustomerMonth               period[M]
purchase_frequency              int64
total_spend                   float64
avg_order_value               float64
unique_items                    int64
days_since_last_purchase      float64
is_first_month                   bool
is_churn                        int64
dtype: object


,Customer ID,CustomerMonth,purchase_frequency,total_spend,avg_order_value,unique_items,days_since_last_purchase,is_first_month,is_churn
0,12346,2010-03,1,27.05,27.05,5,NaN,True,1
1,12346,2010-06,1,142.31,142.31,19,118.0,False,1
2,12346,2011-01,1,149.76,149.76,1,203.0,False,1
3,12347,2010-10,1,611.53,611.53,40,NaN,True,0
4,12347,2010-12,1,711.79,711.79,31,37.0,False,0
5,12347,2011-01,1,475.39,475.39,29,49.0,False,0
6,12347,2011-04,1,536.41,536.41,24,70.0,False,0
7,12347,2011-06,1,382.52,382.52,18,63.0,False,0
8,12347,2011-08,1,584.91,584.91,22,53.0,False,0
9,12348,2010-09,1,221.16,221.16,19,NaN,True,0


In [14]:
## save to new parquest file
customer_month_features.to_parquet('../data/online_retail_ii_customer_month_features.parquet', index=False)

print("Saved to data data/online_retail_ii_customer_month_features.parquet")

Saved to data data/online_retail_ii_customer_month_features.parquet
